In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/reddit_wsb.csv')

In [2]:
df = df.drop('score', axis=1).drop('url', axis=1).drop('comms_num', axis=1).drop('id', axis=1).drop('timestamp', axis=1)
df['eastern'] = pd.to_datetime(df['created'], unit='s').dt.tz_localize('Asia/Shanghai').dt.tz_convert('US/Eastern')
df = df.drop('created', axis=1)
df['body'] = df['body'].fillna('')
df['text'] = df['title'] + '\n' + df['body']
df = df.drop("title", axis=1).drop("body", axis=1)

In [3]:
import re

# Removes URLs, reddit user mentions, and goofy whitespace stuff
def clean_text(text):
  text = re.sub(r'http\S+|www\S+', '', text)
  text = re.sub(r'u/\w+', '', text)
  text = re.sub(r'\s+', ' ', text)
  return text.strip()

In [4]:
df['text'] = df['text'].apply(clean_text)
df = df[df['text'].str.strip().astype(bool)]

In [5]:
print("Earliest:", df['eastern'].min())
print("Latest:", df['eastern'].max())

Earliest: 2020-09-28 12:46:56-04:00
Latest: 2021-08-15 18:26:20-04:00


In [6]:
df.head()

,eastern,text
0,2021-01-28 06:37:41-05:00,"It's not about the money, it's about sending a..."
1,2021-01-28 06:32:10-05:00,Math Professor Scott Steiner says the numbers ...
2,2021-01-28 06:30:35-05:00,Exit the system The CEO of NASDAQ pushed to ha...
3,2021-01-28 06:28:57-05:00,NEW SEC FILING FOR GME! CAN SOMEONE LESS RETAR...
4,2021-01-28 06:26:56-05:00,"Not to distract from GME, just thought our AMC..."


In [7]:
from pytickersymbols import PyTickerSymbols

pts = PyTickerSymbols()

spy_5_tickers = pts.get_sp_500_nyc_yahoo_tickers()
spy_6_tickers = pts.get_sp_600_nyc_yahoo_tickers()
dow_tickers = pts.get_dow_jones_nyc_yahoo_tickers()
nasdaq_tickers = pts.get_nasdaq_100_nyc_yahoo_tickers()

unique_tickers = set(spy_5_tickers + spy_6_tickers + dow_tickers + nasdaq_tickers)
unique_tickers = {t for t in unique_tickers if not t[0].isdigit()}

ordered_tickers = sorted(unique_tickers)

print(ordered_tickers)
print(len(ordered_tickers))

['A', 'AAL', 'AAP', 'AAPL', 'ABBV', 'ABC', 'ABMD', 'ABNB', 'ABT', 'ACGL', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'ADSK', 'ADTN', 'AEE', 'AEP', 'AES', 'AFL', 'AIG', 'AIZ', 'AJG', 'AKAM', 'ALB', 'ALGN', 'ALK', 'ALL', 'ALLE', 'AMAT', 'AMCCF', 'AMCR', 'AMD', 'AME', 'AMGN', 'AMP', 'AMT', 'AMZN', 'ANET', 'ANSS', 'AON', 'AOS', 'APA', 'APD', 'APH', 'APTV', 'ARE', 'ASML', 'ASMLF', 'ATO', 'ATVI', 'AVB', 'AVGO', 'AVY', 'AWK', 'AXP', 'AZN', 'AZNCF', 'AZO', 'BA', 'BAC', 'BAIDF', 'BALL', 'BALY', 'BAX', 'BBWI', 'BBY', 'BDX', 'BEN', 'BF-B', 'BIDU', 'BIIB', 'BIO', 'BK', 'BKNG', 'BKR', 'BLK', 'BLL', 'BMY', 'BMYMP', 'BOAPL', 'BR', 'BRK-B', 'BRO', 'BSX', 'BWA', 'BXP', 'C', 'CAG', 'CAH', 'CARR', 'CAT', 'CB', 'CBRE', 'CCI', 'CCL', 'CDAY', 'CDNS', 'CDW', 'CE', 'CEG', 'CF', 'CFG', 'CHD', 'CHRW', 'CHTR', 'CI', 'CINF', 'CL', 'CLX', 'CMA', 'CMCSA', 'CME', 'CMG', 'CMI', 'CMS', 'CMS-PB', 'CNC', 'CNP', 'CNP-PB', 'COF', 'COO', 'COP', 'COST', 'CPB', 'CPRT', 'CPT', 'CRL', 'CRM', 'CRWD', 'CSCO', 'CSGP', 'CSX', 'CTAS', 'CT

In [8]:
tickers_to_names_dirty = {t: pts.get_stock_name_by_yahoo_symbol(t) for t in ordered_tickers}

In [9]:
suffixes = [
  r'Holding',
  r'N\.V\.',
  r'Holding N\.V\.',
  r'Properties',
  r'Bancshares',
  r'Companies',
  r'Limited',
  r'Realty',
  r'Inc\.?',
  r'Ltd\.?',
  r'Corporation',
  r'Company',
  r'Group',
  r'& Co\.?',
  r'Co',
  r'PLC',
]

suffixes = sorted(suffixes, key=len, reverse=True)

removal = r'(?:\s|,)*(' + '|'.join(suffixes) + r')(\s|,|$)+'
print(repr(removal))

def clean_name(name):
  # Could not get this fucking regex to work
  if name == 'ASML Holding N.V.':
    return 'ASML'
  prev = None
  while prev != name:
    prev = name
    name = re.sub(removal, '', name, flags=re.IGNORECASE).strip()
  return name

'(?:\\s|,)*(Holding N\\.V\\.|Corporation|Properties|Bancshares|Companies|Holding|Limited|Company|& Co\\.?|N\\.V\\.|Realty|Inc\\.?|Ltd\\.?|Group|PLC|Co)(\\s|,|$)+'


In [10]:
# Redditors don't use yf endings and partial string matches are fine
tickers_to_names = {t: clean_name(n) for t, n in tickers_to_names_dirty.items()}

from collections import defaultdict

# Build reverse mapping
value_to_keys = defaultdict(list)
for k, v in tickers_to_names.items():
  value_to_keys[v].append(k)

associations = {}

successful_associations = 0
failed_associations = 0
successful_resolutions = 0
failed_resolutions = 0
remaining_names = []

goofy_ahh_endings = ('PB','PF','PI', 'PK','PR','A','B','C','F', 'WI', 'MP', 'UKY', 'LV', 'RP', '-PB', 'GP')

for name, tickers in value_to_keys.items():
  n = len(tickers)
  if n == 0:
    raise Exception('how did we get here')
  elif n == 1:
    associations[tickers[0]] = [name, tickers[0]]
    successful_associations += 1
  else:
    primary_tickers = [t for t in tickers if not t.upper().endswith(goofy_ahh_endings)]
    secondary_tickers = [t for t in tickers if t.upper().endswith(goofy_ahh_endings)]
    if len(primary_tickers) == 1 and n - len(secondary_tickers) == 1:
      associations[primary_tickers[0]] = [name, *tickers]
      successful_associations += 1
      successful_resolutions += 1
    elif len(primary_tickers) > 0 and len(secondary_tickers) > 0:
      value_to_keys[name] = primary_tickers
    else:
      failed_resolutions += 1
      failed_associations += 1
      remaining_names.append(name)

print(f'Successful associations: {successful_associations}')
print(f'Failed associations: {failed_associations}')
print(f'Successful resolutions: {successful_resolutions}')
print(f'Failed resolutions: {failed_resolutions}')
print(associations)
for n in remaining_names:
  print(f'{n}: {value_to_keys[n]}')

Successful associations: 502
Failed associations: 16
Successful resolutions: 24
Failed resolutions: 16
{'A': ['Agilent Technologies', 'A'], 'AAL': ['American Airlines', 'AAL'], 'AAP': ['Advance Auto Parts', 'AAP'], 'AAPL': ['Apple', 'AAPL'], 'ABBV': ['AbbVie', 'ABBV'], 'ABC': ['AmerisourceBergen', 'ABC'], 'ABMD': ['Abiomed', 'ABMD'], 'ABNB': ['Airbnb', 'ABNB'], 'ABT': ['Abbott Laboratories', 'ABT'], 'ACGL': ['Arch Capital', 'ACGL'], 'ACN': ['Accenture', 'ACN'], 'ADBE': ['Adobe', 'ADBE'], 'ADI': ['Analog Devices', 'ADI'], 'ADM': ['Archer-Daniels-Midland', 'ADM'], 'ADP': ['Automatic Data Processing', 'ADP'], 'ADSK': ['Autodesk', 'ADSK'], 'ADTN': ['ADTRAN', 'ADTN'], 'AEE': ['Ameren', 'AEE'], 'AEP': ['American Electric Power', 'AEP'], 'AES': ['The AES', 'AES'], 'AFL': ['Aflac', 'AFL'], 'AIG': ['American International', 'AIG'], 'AIZ': ['Assurant', 'AIZ'], 'AJG': ['Arthur J. Gallagher', 'AJG'], 'AKAM': ['Akamai Technologies', 'AKAM'], 'ALB': ['Albemarle', 'ALB'], 'ALGN': ['Align Technology',

In [ ]:
# Just hardcoding the rest

associations['FISV'] = ['Fiserv', 'FI', 'FISV']
associations['FRCB'] = ['First Republic Bank', 'FRCB', 'FRC']
associations['GOOG'] = ['Alphabet', 'GOOG', 'GOOGL']
associations['HBAN'] = ['Huntington', 'Huntington Bancshares', 'HBAN', 'HBANN']
associations['LHX'] = ['L3Harris', 'HRS', 'LHX']
associations['IP'] = ['International Paper', 'INPAP', 'IP']
associations['NEE'] = ['NextEra Energy', 'NEE', 'NEEXU']
associations['O'] = ['Realty Income', 'Realty Income Corp', 'O',]
associations['UHT'] = ['Universal Health Realty', 'UHT']
associations['PSKY'] = ['Paramount', 'Paramount Skydance', 'PARA', 'PARAA', 'PSKY']
associations['RF'] = ['Regions Financial', 'RF', 'RF-PB']
associations['UHS'] = ['Universal Health Services', 'UHID', 'UHS']
associations['WTW'] = ['WTW Public', 'Willis Towers Watson', 'WTW', 'WLTW']
associations['XOM'] = ['Exxon Mobil', 'XOM', 'XON']

# Confused for real english words or common abbreviations
del associations['A']
del associations['COO']
del associations['BRO']
del associations['L']

# Ignoring BALL and CEG/STZ because source data is wrong
# Ignoring Healthpeak because there are a lot of conflicts with these tickers and it'll convolute the data

# Manually adding meme stocks

associations['GME'] = ['GME', 'GameStop']
associations['AMC'] = ['AMC']
associations['BB'] = ['BB', 'BlackBerry']
associations['KOSS'] = ['KOSS', 'Koss']
associations['NOK'] = ['NOK', 'Nokia']
associations['EXPR'] = ['EXPR', 'Express']
associations['SNDL'] = ['SNDL', 'SunDial']
associations['BBBY'] = ['BBBY', 'Bed Bath Beyond', 'Bed Bath & Beyond']
associations['PLTR'] = ['PLTR', 'Palantir']
associations['CLOV'] = ['CLOV', 'Clover']
associations['SPCE'] = ['SPCE', 'Virgin Galactic']
associations['VIR'] = ['VIR']
associations['FUBO'] = ['FUBO', 'Fubo', 'Fubotv']
associations['BYND'] = ['BYND', 'Beyond']
associations['WISH'] = ['WISH', 'ContextLogic']
associations['CVNA'] = ['CVNA', 'Carvana']
associations['UPST'] = ['UPST', 'Upstart']
associations['NDAQ'] += ['QQQ', 'QQQM']
associations['QQQ'] = associations['NDAQ']
del associations['NDAQ']
associations['SPY'] = ['SPY', 'SPX', 'SP500', 'S&P500']
print(associations)

{'AAL': ['American Airlines', 'AAL'], 'AAP': ['Advance Auto Parts', 'AAP'], 'AAPL': ['Apple', 'AAPL'], 'ABBV': ['AbbVie', 'ABBV'], 'ABC': ['AmerisourceBergen', 'ABC'], 'ABMD': ['Abiomed', 'ABMD'], 'ABNB': ['Airbnb', 'ABNB'], 'ABT': ['Abbott Laboratories', 'ABT'], 'ACGL': ['Arch Capital', 'ACGL'], 'ACN': ['Accenture', 'ACN'], 'ADBE': ['Adobe', 'ADBE'], 'ADI': ['Analog Devices', 'ADI'], 'ADM': ['Archer-Daniels-Midland', 'ADM'], 'ADP': ['Automatic Data Processing', 'ADP'], 'ADSK': ['Autodesk', 'ADSK'], 'ADTN': ['ADTRAN', 'ADTN'], 'AEE': ['Ameren', 'AEE'], 'AEP': ['American Electric Power', 'AEP'], 'AES': ['The AES', 'AES'], 'AFL': ['Aflac', 'AFL'], 'AIG': ['American International', 'AIG'], 'AIZ': ['Assurant', 'AIZ'], 'AJG': ['Arthur J. Gallagher', 'AJG'], 'AKAM': ['Akamai Technologies', 'AKAM'], 'ALB': ['Albemarle', 'ALB'], 'ALGN': ['Align Technology', 'ALGN'], 'ALK': ['Alaska Air', 'ALK'], 'ALL': ['The Allstate', 'ALL'], 'ALLE': ['Allegion', 'ALLE'], 'AMAT': ['Applied Materials', 'AMAT']

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas()

sep = r'(?:^|[\s.,;:!?()\[\]"\'])'
end_sep = r'(?=$|[\s.,;:!?()\[\]"\'])'
uppercase_only = [
  'ARE',
  'HAS',
  'IT',
  'NOW',
  'ON',
  'SEE',
  'LOW',
  'SO',
  'HES',
  'T',
  'FAST',
  'WISH',
  'SEE',
  'PPL',
  'ICE',
  'BEN',
  'K',
  'KIM',
  'DTE',
  'D',
  'KEYS',
  'COST',
  'POOL',
  'ALL',
  'WELL',
  'O',
]

ticker_patterns = {}
for ticker, names in associations.items():
  name_patterns = []
  for name in names:
    escaped = re.escape(name)
    if name.upper() in uppercase_only:
      name_patterns.append(escaped)
    else:
      name_patterns.append(f'(?i:{escaped})')
  pattern = sep + '(' + '|'.join(name_patterns) + ')' + end_sep
  ticker_patterns[ticker] = re.compile(pattern)

def find_tickers(text):
  mentioned = [ticker for ticker, pattern in ticker_patterns.items() if pattern.search(text)]
  return mentioned

df_sample = df.head(100)
df_sample['mentioned_tickers'] = df_sample['text'].progress_apply(find_tickers)


/Users/OwenScott/.pyenv/versions/3.13.5/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 100/100 [00:01<00:00, 89.02it/s]
/var/folders/2s/vhdt8ph920d695tdxy4n9w9h0000gn/T/ipykernel_27887/3026086501.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sample['mentioned_tickers'] = df_sample['text'].progress_apply(find_tickers)


In [13]:
df['mentioned_tickers'] = df['text'].progress_apply(find_tickers)

100%|██████████| 53183/53183 [07:29<00:00, 118.44it/s]


In [14]:
df = df[df['mentioned_tickers'].map(len) > 0]
print(len(df))
df.head()

30904


,eastern,text,mentioned_tickers
1,2021-01-28 06:32:10-05:00,Math Professor Scott Steiner says the numbers ...,[GME]
2,2021-01-28 06:30:35-05:00,Exit the system The CEO of NASDAQ pushed to ha...,"[GME, QQQ]"
3,2021-01-28 06:28:57-05:00,NEW SEC FILING FOR GME! CAN SOMEONE LESS RETAR...,[GME]
4,2021-01-28 06:26:56-05:00,"Not to distract from GME, just thought our AMC...","[GME, AMC]"
6,2021-01-28 06:26:27-05:00,SHORT STOCK DOESN'T HAVE AN EXPIRATION DATE He...,[T]


In [16]:
df.to_parquet('../data/reddit_wsb_cache.parquet', engine='pyarrow', compression='gzip')